In [ ]:
# Cell 1: Install required dependencies
!pip install -q kaggle pandas numpy scikit-learn opencv-python matplotlib \
               albumentations scikit-image scipy torch torchvision tqdm

# Cell 2: Set up Kaggle credentials
import os, json, pathlib
from google.colab import files

uploaded = files.upload()
if "kaggle.json" not in uploaded:
    raise RuntimeError("kaggle.json not uploaded.")

KAGGLE_DIR = os.path.join(os.path.expanduser("~"), ".kaggle")
os.makedirs(KAGGLE_DIR, exist_ok=True)
kaggle_path = os.path.join(KAGGLE_DIR, "kaggle.json")
with open(kaggle_path, "wb") as f:
    f.write(uploaded["kaggle.json"])
os.chmod(kaggle_path, 0o600)

# Cell 3: Download and extract CBIS-DDSM dataset
from pathlib import Path

BASE_DIR = Path("/content")
DATA_ROOT = BASE_DIR / "data"
DATA_ROOT.mkdir(parents=True, exist_ok=True)

KAGGLE_DATASET_MAIN = "awsaf49/cbis-ddsm-breast-cancer-image-dataset"

CBIS_MAIN_DIR = DATA_ROOT / "cbis_ddsm_main"
CBIS_MAIN_DIR.mkdir(parents=True, exist_ok=True)

!kaggle datasets download -d {KAGGLE_DATASET_MAIN} -p "{CBIS_MAIN_DIR}" --force

import zipfile, os

zip_files = [f for f in os.listdir(CBIS_MAIN_DIR) if f.endswith(".zip")]
for z in zip_files:
    z_path = CBIS_MAIN_DIR / z
    with zipfile.ZipFile(z_path, "r") as zip_ref:
        zip_ref.extractall(CBIS_MAIN_DIR)

# Cell 4: Import libraries and set random seed
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import pandas as pd
import cv2
from skimage import exposure
from scipy.signal import wiener
import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, confusion_matrix, classification_report, roc_curve

import matplotlib.pyplot as plt
from tqdm.auto import tqdm

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"

# Cell 5: Load and preprocess dataset metadata
csv_dir = CBIS_MAIN_DIR / "csv"

mass_train = pd.read_csv(csv_dir / "mass_case_description_train_set.csv")
mass_test  = pd.read_csv(csv_dir / "mass_case_description_test_set.csv")
calc_train = pd.read_csv(csv_dir / "calc_case_description_train_set.csv")
calc_test  = pd.read_csv(csv_dir / "calc_case_description_test_set.csv")

PATHOLOGY_MAP = {
    "BENIGN": 0,
    "BENIGN_WITHOUT_CALLBACK": 0,
    "MALIGNANT": 1
}

def prepare_df(df, lesion_type):
    df = df.copy()
    df["label"] = df["pathology"].map(PATHOLOGY_MAP)
    df["lesion_type"] = lesion_type
    return df

mass_train = prepare_df(mass_train, "MASS")
mass_test  = prepare_df(mass_test, "MASS")
calc_train = prepare_df(calc_train, "CALC")
calc_test  = prepare_df(calc_test, "CALC")

all_df = pd.concat([mass_train, mass_test, calc_train, calc_test], ignore_index=True)
all_df = all_df.dropna(subset=["label"]).reset_index(drop=True)

PATH_COL = "image file path"
ROI_COL  = "ROI mask file path"

if PATH_COL not in all_df.columns:
    raise RuntimeError("Adjust PATH_COL/ROI_COL to match CSV columns.")

def build_abs_path(rel_path):
    return str(CBIS_MAIN_DIR / rel_path)

all_df["img_path"] = all_df[PATH_COL].apply(build_abs_path)
all_df["mask_path"] = all_df[ROI_COL].apply(build_abs_path)

# Cell 6: Image preprocessing functions
IMG_SIZE = 512

def resize_with_pad(img, size=IMG_SIZE):
    h, w = img.shape[:2]
    scale = size / max(h, w)
    nh, nw = int(h * scale), int(w * scale)
    img_resized = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_AREA)
    top = (size - nh) // 2
    bottom = size - nh - top
    left = (size - nw) // 2
    right = size - nw - left
    img_padded = cv2.copyMakeBorder(
        img_resized, top, bottom, left, right,
        borderType=cv2.BORDER_CONSTANT, value=0
    )
    return img_padded

def wiener_filter(img):
    out = wiener(img, (3, 3))
    out = np.clip(out, 0.0, 1.0)
    return out

def apply_clahe(img):
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    img_clahe = clahe.apply((img * 255).astype(np.uint8))
    img_clahe = img_clahe.astype(np.float32) / 255.0
    return img_clahe

def breast_region_mask(img):
    img_u8 = (img * 255).astype(np.uint8)
    _, th = cv2.threshold(img_u8, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    if np.mean(img_u8[th == 255]) < np.mean(img_u8[th == 0]):
        th = cv2.bitwise_not(th)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    mask = cv2.morphologyEx(th, cv2.MORPH_CLOSE, kernel, iterations=2)
    mask = (mask > 0).astype(np.float32)
    return mask

def apply_breast_crop(img):
    mask = breast_region_mask(img)
    return img * mask

def normalize_zscore(img):
    m = img.mean()
    s = img.std() + 1e-6
    img_z = (img - m) / s
    return img_z

def histogram_standardize(img):
    p2, p98 = np.percentile(img, (2, 98))
    img_rescaled = exposure.rescale_intensity(img, in_range=(p2, p98))
    return img_rescaled

def full_preprocess(img_raw):
    img = img_raw.astype(np.float32)
    if img.max() > 1.0:
        img /= 255.0
    img = resize_with_pad(img, IMG_SIZE)
    img = wiener_filter(img)
    img = apply_clahe(img)
    img = apply_breast_crop(img)
    img = histogram_standardize(img)
    img = np.clip(img, 0.0, 1.0)
    img = normalize_zscore(img)
    return img

# Cell 7: Data augmentation pipelines
train_aug = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.ShiftScaleRotate(
        shift_limit=0.10, scale_limit=0.1, rotate_limit=15,
        border_mode=cv2.BORDER_CONSTANT, value=0, p=0.7
    ),
    A.RandomBrightnessContrast(brightness_limit=0.1, contrast_limit=0.1, p=0.5),
    A.GaussNoise(var_limit=(0.0, 0.01), p=0.3),
    A.GaussianBlur(blur_limit=(3, 5), p=0.3),
    ToTensorV2()
])

val_aug = A.Compose([
    ToTensorV2()
])

# Cell 8: Custom PyTorch Dataset (CBISDataset)
class CBISDataset(Dataset):
    def __init__(self, df: pd.DataFrame, augment=None):
        self.df = df.reset_index(drop=True)
        self.augment = augment

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = row["img_path"]
        mask_path = row["mask_path"]
        label = int(row["label"])

        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            raise RuntimeError(f"Cannot read image: {img_path}")
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        if mask is None:
            mask = np.zeros_like(img, dtype=np.uint8)

        img = img.astype(np.float32)
        mask = (mask > 0).astype(np.float32)

        img = resize_with_pad(img, IMG_SIZE)
        mask = resize_with_pad(mask, IMG_SIZE)

        img = img / 255.0
        img = wiener_filter(img)
        img = apply_clahe(img)
        img = apply_breast_crop(img)
        img = histogram_standardize(img)
        img = np.clip(img, 0.0, 1.0)
        img = normalize_zscore(img)

        if self.augment is not None:
            augmented = self.augment(image=img, mask=mask)
            img_t = augmented["image"]
            mask_t = augmented["mask"].unsqueeze(0)
        else:
            img_t = torch.from_numpy(img).unsqueeze(0).float()
            mask_t = torch.from_numpy(mask).unsqueeze(0).float()

        label_t = torch.tensor(label, dtype=torch.long)
        return img_t, mask_t, label_t

# Cell 9: Model Architectures (ViT, MemoryBank, Segmentation)
import math

class PatchEmbed(nn.Module):
    def __init__(self, img_size=512, patch_size=16, in_chans=1, embed_dim=768):
        super().__init__()
        self.img_size = img_size
        self.patch_size = patch_size
        self.grid_size = img_size // patch_size
        self.num_patches = self.grid_size * self.grid_size
        self.proj = nn.Conv2d(in_chans, embed_dim, kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        x = self.proj(x)
        x = x.flatten(2).transpose(1, 2)
        return x

class MLP(nn.Module):
    def __init__(self, in_features, hidden_features=None, out_features=None, drop=0.0):
        super().__init__()
        out_features = out_features or in_features
        hidden_features = hidden_features or in_features
        self.fc1 = nn.Linear(in_features, hidden_features)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(hidden_features, out_features)
        self.drop = nn.Dropout(drop)

    def forward(self, x):
        x = self.fc1(x)
        x = self.act(x)
        x = self.drop(x)
        x = self.fc2(x)
        x = self.drop(x)
        return x

class TransformerEncoderBlock(nn.Module):
    def __init__(self, dim, num_heads=12, mlp_ratio=4.0, drop=0.0, attn_drop=0.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, num_heads, dropout=attn_drop, batch_first=True)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = MLP(dim, int(dim * mlp_ratio), drop=drop)

    def forward(self, x):
        x_norm = self.norm1(x)
        attn_out, _ = self.attn(x_norm, x_norm, x_norm)
        x = x + attn_out
        x = x + self.mlp(self.norm2(x))
        return x

class ViTEncoder(nn.Module):
    def __init__(
        self,
        img_size=512,
        patch_size=16,
        in_chans=1,
        embed_dim=768,
        depth=12,
        num_heads=12,
        mlp_ratio=4.0,
        drop_rate=0.0,
        attn_drop_rate=0.0
    ):
        super().__init__()
        self.patch_embed = PatchEmbed(img_size, patch_size, in_chans, embed_dim)
        num_patches = self.patch_embed.num_patches
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches + 1, embed_dim))
        self.pos_drop = nn.Dropout(drop_rate)
        self.blocks = nn.ModuleList([
            TransformerEncoderBlock(
                embed_dim,
                num_heads=num_heads,
                mlp_ratio=mlp_ratio,
                drop=drop_rate,
                attn_drop=attn_drop_rate
            )
            for _ in range(depth)
        ])
        self.norm = nn.LayerNorm(embed_dim)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        nn.init.trunc_normal_(self.cls_token, std=0.02)

    def forward(self, x):
        B = x.shape[0]
        x = self.patch_embed(x)
        cls_tokens = self.cls_token.expand(B, -1, -1)
        x = torch.cat((cls_tokens, x), dim=1)
        x = x + self.pos_embed
        x = self.pos_drop(x)
        for blk in self.blocks:
            x = blk(x)
        x = self.norm(x)
        cls = x[:, 0]
        patches = x[:, 1:]
        h = w = int(math.sqrt(patches.shape[1]))
        patches = patches.view(B, h, w, -1)
        return cls, patches

class MemoryBank(nn.Module):
    def __init__(self, dim=768, device="cuda"):
        super().__init__()
        self.register_buffer("keys", torch.empty(0, dim))
        self.register_buffer("values", torch.empty(0, dim))
        self.device = device

    @torch.no_grad()
    def add(self, k, v):
        k = k.detach().to(self.keys.device)
        v = v.detach().to(self.values.device)
        self.keys = torch.cat([self.keys, k], dim=0)
        self.values = torch.cat([self.values, v], dim=0)

    def forward(self, q, topk=5):
        if self.keys.numel() == 0:
            return torch.zeros_like(q), torch.full((q.size(0), topk), -1, dtype=torch.long, device=q.device)
        q_norm = F.normalize(q, dim=-1)
        k_norm = F.normalize(self.keys, dim=-1)
        sims = q_norm @ k_norm.t()
        topk_sims, topk_idx = sims.topk(topk, dim=-1)
        alpha = F.softmax(topk_sims, dim=-1)
        v = self.values[topk_idx]
        retrieved = torch.sum(alpha.unsqueeze(-1) * v, dim=1)
        return retrieved, topk_idx

class CrossAttentionFusion(nn.Module):
    def __init__(self, dim=768, num_heads=8, drop=0.1):
        super().__init__()
        self.norm_q = nn.LayerNorm(dim)
        self.norm_kv = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, num_heads, dropout=drop, batch_first=True)
        self.mlp = MLP(dim, int(dim * 4), drop=drop)
        self.norm_out = nn.LayerNorm(dim)

    def forward(self, visual_feat, retrieved_feat=None):
        B, D = visual_feat.shape
        q = visual_feat.unsqueeze(1)
        tokens = [visual_feat]
        if retrieved_feat is not None:
            tokens.append(retrieved_feat)
        kv = torch.stack(tokens, dim=1)
        q_n = self.norm_q(q)
        kv_n = self.norm_kv(kv)
        attn_out, _ = self.attn(q_n, kv_n, kv_n)
        fused = q + attn_out
        fused = fused + self.mlp(self.norm_out(fused))
        return fused.squeeze(1)

class SimpleSegDecoder(nn.Module):
    def __init__(self, in_dim=768, base_channels=256, out_channels=1):
        super().__init__()
        self.proj = nn.Conv2d(in_dim, base_channels, 1)
        self.up1 = nn.Sequential(
            nn.Conv2d(base_channels, base_channels, 3, padding=1),
            nn.BatchNorm2d(base_channels),
            nn.ReLU(True)
        )
        self.up2 = nn.Sequential(
            nn.Conv2d(base_channels, base_channels // 2, 3, padding=1),
            nn.BatchNorm2d(base_channels // 2),
            nn.ReLU(True)
        )
        self.up3 = nn.Sequential(
            nn.Conv2d(base_channels // 2, base_channels // 4, 3, padding=1),
            nn.BatchNorm2d(base_channels // 4),
            nn.ReLU(True)
        )
        self.out_conv = nn.Conv2d(base_channels // 4, out_channels, 1)

    def forward(self, patch_feats, global_token=None):
        B, H, W, D = patch_feats.shape
        x = patch_feats.permute(0, 3, 1, 2)
        if global_token is not None:
            g = global_token.unsqueeze(-1).unsqueeze(-1)
            g = F.interpolate(g, size=(H, W), mode="nearest")
            x = x + g
        x = self.proj(x)
        x = F.interpolate(x, scale_factor=2, mode="bilinear", align_corners=False)
        x = self.up1(x)
        x = F.interpolate(x, scale_factor=2, mode="bilinear", align_corners=False)
        x = self.up2(x)
        x = F.interpolate(x, scale_factor=2, mode="bilinear", align_corners=False)
        x = self.up3(x)
        x = F.interpolate(x, scale_factor=2, mode="bilinear", align_corners=False)
        logits = self.out_conv(x)
        return logits

class ViTMultiRAGNet(nn.Module):
    def __init__(self, img_size=512, num_classes=2):
        super().__init__()
        self.vit = ViTEncoder(
            img_size=img_size,
            patch_size=16,
            in_chans=1,
            embed_dim=768,
            depth=12,
            num_heads=12
        )
        self.fusion = CrossAttentionFusion(dim=768, num_heads=8, drop=0.1)
        self.classifier = nn.Sequential(
            nn.LayerNorm(768),
            nn.Linear(768, 384),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(384, num_classes)
        )
        self.seg_decoder = SimpleSegDecoder(in_dim=768, base_channels=256, out_channels=1)

    def forward(self, x_img, memory_bank=None, topk=5):
        cls, patches = self.vit(x_img)
        rag_feat, idx = (None, None)
        if memory_bank is not None:
            rag_feat, idx = memory_bank(cls.detach(), topk=topk)
        fused = self.fusion(cls, rag_feat)
        logits_cls = self.classifier(fused)
        logits_seg = self.seg_decoder(patches, global_token=fused)
        return logits_cls, logits_seg, idx

# Cell 1: Install required dependencies0
class DiceLoss(nn.Module):
    def __init__(self, eps=1e-6):
        super().__init__()
        self.eps = eps

    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)
        num = 2 * torch.sum(probs * targets) + self.eps
        den = torch.sum(probs + targets) + self.eps
        return 1.0 - num / den

def dice_bce_loss(logits, targets, bce_weight=0.5):
    bce = F.binary_cross_entropy_with_logits(logits, targets)
    d   = DiceLoss()(logits, targets)
    return bce_weight * bce + (1.0 - bce_weight) * d

def ce_with_consistency(logits, labels, alpha=0.1):
    ce = F.cross_entropy(logits, labels)
    probs = F.softmax(logits, dim=-1)
    entropy = -torch.sum(probs * torch.log(probs + 1e-6), dim=-1).mean()
    return ce + alpha * entropy

# Cell 1: Install required dependencies1
def train_segmentation(model, train_loader, val_loader, device, epochs=200, lr=1e-4):
    model = model.to(device)
    params = list(model.vit.parameters()) + list(model.seg_decoder.parameters())
    optimizer = torch.optim.AdamW(params, lr=lr, weight_decay=0.01)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    best_val = float("inf")
    patience = 20
    no_improve = 0

    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        for imgs, masks, labels in tqdm(train_loader, desc=f"[Seg] {epoch+1}/{epochs}"):
            imgs, masks = imgs.to(device), masks.to(device)
            optimizer.zero_grad()
            _, logits_seg, _ = model(imgs, memory_bank=None)
            loss = dice_bce_loss(logits_seg, masks, bce_weight=0.5)
            loss.backward()
            nn.utils.clip_grad_norm_(params, max_norm=1.0)
            optimizer.step()
            total_loss += loss.item() * imgs.size(0)

        scheduler.step()
        train_loss = total_loss / len(train_loader.dataset)

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for imgs, masks, labels in val_loader:
                imgs, masks = imgs.to(device), masks.to(device)
                _, logits_seg, _ = model(imgs, memory_bank=None)
                loss = dice_bce_loss(logits_seg, masks, bce_weight=0.5)
                val_loss += loss.item() * imgs.size(0)
        val_loss /= len(val_loader.dataset)

        print(f"[Seg] Epoch {epoch+1}: train {train_loss:.4f}, val {val_loss:.4f}")

        if val_loss < best_val:
            best_val = val_loss
            no_improve = 0
            torch.save(model.state_dict(), "best_segmentation.pth")
        else:
            no_improve += 1
            if no_improve >= patience:
                print("Early stopping segmentation.")
                break

@torch.no_grad()
def build_memory_bank(model, loader, device, dim=768):
    model = model.to(device)
    model.eval()
    memory = MemoryBank(dim=dim, device=device)
    for imgs, masks, labels in tqdm(loader, desc="Memory bank"):
        imgs = imgs.to(device)
        cls, _ = model.vit(imgs)
        memory.add(cls, cls)
    return memory

def train_classification(model, memory_bank, train_loader, val_loader, device, epochs=100, lr=1e-4):
    model = model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    best_auc = 0.0
    patience = 15
    no_improve = 0

    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        all_logits = []
        all_labels = []

        for imgs, masks, labels in tqdm(train_loader, desc=f"[Cls] {epoch+1}/{epochs}"):
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            logits_cls, _, _ = model(imgs, memory_bank=memory_bank, topk=5)
            loss = ce_with_consistency(logits_cls, labels, alpha=0.1)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            total_loss += loss.item() * imgs.size(0)
            all_logits.append(torch.softmax(logits_cls, dim=-1)[:, 1].detach().cpu())
            all_labels.append(labels.detach().cpu())

        scheduler.step()
        train_loss = total_loss / len(train_loader.dataset)
        train_logits = torch.cat(all_logits).numpy()
        train_labels = torch.cat(all_labels).numpy()
        train_auc = roc_auc_score(train_labels, train_logits)

        model.eval()
        v_logits = []
        v_labels = []
        v_loss = 0.0
        with torch.no_grad():
            for imgs, masks, labels in val_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                logits_cls, _, _ = model(imgs, memory_bank=memory_bank, topk=5)
                loss = ce_with_consistency(logits_cls, labels, alpha=0.1)
                v_loss += loss.item() * imgs.size(0)
                v_logits.append(torch.softmax(logits_cls, dim=-1)[:, 1].cpu())
                v_labels.append(labels.cpu())
        v_loss /= len(val_loader.dataset)
        v_logits = torch.cat(v_logits).numpy()
        v_labels = torch.cat(v_labels).numpy()
        val_auc = roc_auc_score(v_labels, v_logits)

        print(f"[Cls] Epoch {epoch+1}: train_loss {train_loss:.4f}, train_auc {train_auc:.4f}, "
              f"val_loss {v_loss:.4f}, val_auc {val_auc:.4f}")

        if val_auc > best_auc:
            best_auc = val_auc
            no_improve = 0
            torch.save(model.state_dict(), "best_classification.pth")
        else:
            no_improve += 1
            if no_improve >= patience:
                print("Early stopping classification.")
                break

# Cell 1: Install required dependencies2
labels = all_df["label"].values
train_val_idx, test_idx = next(StratifiedKFold(n_splits=5, shuffle=True, random_state=42).split(all_df, labels))
train_val_df = all_df.iloc[train_val_idx].reset_index(drop=True)
test_df      = all_df.iloc[test_idx].reset_index(drop=True)

y_tv = train_val_df["label"].values
tv_idx, val_idx = next(StratifiedKFold(n_splits=10, shuffle=True, random_state=42).split(train_val_df, y_tv))
train_df = train_val_df.iloc[tv_idx].reset_index(drop=True)
val_df   = train_val_df.iloc[val_idx].reset_index(drop=True)

BATCH_SEG = 8
BATCH_CLS = 16

train_seg_ds = CBISDataset(train_df, augment=train_aug)
val_seg_ds   = CBISDataset(val_df,   augment=val_aug)
test_ds      = CBISDataset(test_df,  augment=val_aug)

train_seg_loader = DataLoader(train_seg_ds, batch_size=BATCH_SEG, shuffle=True, num_workers=2)
val_seg_loader   = DataLoader(val_seg_ds,   batch_size=BATCH_SEG, shuffle=False, num_workers=2)

model = ViTMultiRAGNet(img_size=IMG_SIZE, num_classes=2)

train_segmentation(model, train_seg_loader, val_seg_loader, device=device, epochs=200, lr=1e-4)

model.load_state_dict(torch.load("best_segmentation.pth", map_location=device))

memory_bank = build_memory_bank(model, train_seg_loader, device=device)

train_cls_loader = DataLoader(train_seg_ds, batch_size=BATCH_CLS, shuffle=True, num_workers=2)
val_cls_loader   = DataLoader(val_seg_ds,   batch_size=BATCH_CLS, shuffle=False, num_workers=2)

train_classification(model, memory_bank, train_cls_loader, val_cls_loader, device=device, epochs=100, lr=1e-4)

# Cell 1: Install required dependencies3
model.load_state_dict(torch.load("best_classification.pth", map_location=device))
model.to(device)
model.eval()

test_loader = DataLoader(test_ds, batch_size=BATCH_CLS, shuffle=False, num_workers=2)

all_logits = []
all_labels = []
all_preds  = []

with torch.no_grad():
    for imgs, masks, labels in tqdm(test_loader, desc="Test"):
        imgs, labels, masks = imgs.to(device), labels.to(device), masks.to(device)
        logits_cls, logits_seg, _ = model(imgs, memory_bank=memory_bank, topk=5)
        probs = torch.softmax(logits_cls, dim=-1)[:, 1]
        preds = (probs > 0.5).long()
        all_logits.append(probs.cpu())
        all_labels.append(labels.cpu())
        all_preds.append(preds.cpu())

all_logits = torch.cat(all_logits).numpy()
all_labels = torch.cat(all_labels).numpy()
all_preds  = torch.cat(all_preds).numpy()

auc = roc_auc_score(all_labels, all_logits)
acc = (all_preds == all_labels).mean()

print("Test AUC:", auc)
print("Test Accuracy:", acc)
print("\nClassification report:")
print(classification_report(all_labels, all_preds, target_names=["Benign", "Malignant"]))

cm = confusion_matrix(all_labels, all_preds)
print("Confusion matrix:\n", cm)

fpr, tpr, _ = roc_curve(all_labels, all_logits)
plt.figure()
plt.plot(fpr, tpr, label=f"AUC={auc:.3f}")
plt.plot([0,1],[0,1],'k--')
plt.xlabel("FPR")
plt.ylabel("TPR")
plt.title("CBIS-DDSM ROC")
plt.legend()
plt.grid(True)
plt.show()
